# Flight price history

Reads the scraie PostgreSQL database and visualizes how the price of a **single itinerary** changes over time.

Each daily run stores several fare `options` per itinerary, all sharing the same `searched_at` timestamp. For a price-over-time view we collapse those options per search into the **cheapest** fare (and also show the min/max spread).

Set the itinerary you care about in the **Config** cell below (`ITINERARY_ID`). Run the *List itineraries* cell first if you don't know the id.

## Dependencies

Run once if these aren't already installed.

In [ ]:
%pip install -q pandas matplotlib psycopg2-binary python-dotenv

In [ ]:
import os
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
import psycopg2
from dotenv import find_dotenv, load_dotenv

## Config

- `ITINERARY_IDS` — the itineraries to plot, all overlaid on one chart (see the *List itineraries* cell to find ids).
- `FLIGHTS_DB_URI` — Postgres connection string (same var the Go service uses). Loaded from the repo-root `.env` via `python-dotenv`. The pgx-only `default_query_exec_mode` option the Go service appends is stripped, since libpq/psycopg2 rejects it.

In [ ]:
# ---- Edit me -------------------------------------------------------------
ITINERARY_IDS = [1, 2, 3, 4]
# --------------------------------------------------------------------------

# Load the repo-root .env (walks up from the notebook dir to find it).
load_dotenv(find_dotenv(usecwd=True))


def _load_db_uri() -> str:
    uri = os.environ.get("FLIGHTS_DB_URI")
    if not uri:
        raise RuntimeError(
            "FLIGHTS_DB_URI is not set. Add it to the .env at the repo root."
        )
    # The Go service appends a pgx-only option (default_query_exec_mode) that
    # libpq/psycopg2 rejects; drop any such non-libpq params before connecting.
    parts = urlsplit(uri)
    query = [
        (k, v)
        for k, v in parse_qsl(parts.query)
        if k != "default_query_exec_mode"
    ]
    return urlunsplit(parts._replace(query=urlencode(query)))


conn = psycopg2.connect(_load_db_uri())

## List itineraries

Use this to find the `id` to put in `ITINERARY_ID` above.

In [ ]:
_TYPES = {1: "roundtrip", 2: "oneway", 3: "multicity"}
_CLASSES = {1: "economy", 2: "premium_economy", 3: "business", 4: "first"}

itineraries = pd.read_sql(
    """
    SELECT i.id,
           i.departure_id,
           i.arrival_id,
           i.type,
           i.travel_class,
           i.outbound_date,
           i.return_date,
           i.currency,
           i.invalid,
           COUNT(DISTINCT o.searched_at) AS searches,
           MIN(o.searched_at)            AS first_seen,
           MAX(o.searched_at)            AS last_seen
    FROM itineraries i
    LEFT JOIN options o ON o.itinerary_id = i.id
    GROUP BY i.id
    ORDER BY i.id
    """,
    conn,
)
itineraries["type"] = itineraries["type"].map(_TYPES).fillna(itineraries["type"])
itineraries["travel_class"] = (
    itineraries["travel_class"].map(_CLASSES).fillna(itineraries["travel_class"])
)
itineraries

## Load price history for the selected itineraries

# One row per fare option; many options share a searched_at timestamp.
options = pd.read_sql(
    """
    SELECT itinerary_id, searched_at, price, type, total_duration
    FROM options
    WHERE itinerary_id = ANY(%(itins)s)
    ORDER BY itinerary_id, searched_at
    """,
    conn,
    params={"itins": ITINERARY_IDS},
    parse_dates=["searched_at"],
)

if options.empty:
    raise RuntimeError(
        f"No options found for itineraries {ITINERARY_IDS}. "
        "Check the ids in the list above."
    )

# Collapse the options of each search into min / median / max price, per itinerary.
history = (
    options.groupby(["itinerary_id", "searched_at"])["price"]
    .agg(min_price="min", median_price="median", max_price="max", num_options="count")
    .reset_index()
    .sort_values(["itinerary_id", "searched_at"])
)
history

In [ ]:
# One row per fare option; many options share a searched_at timestamp.
options = pd.read_sql(
    """
    SELECT itinerary_id, searched_at, price, type, total_duration
    FROM options
    WHERE itinerary_id = ANY(%(itins)s)
    ORDER BY itinerary_id, searched_at
    """,
    conn,
    params={"itins": ITINERARY_IDS},
    parse_dates=["searched_at"],
)

if options.empty:
    raise RuntimeError(
        f"No options found for itineraries {ITINERARY_IDS}. "
        "Check the ids in the list above."
    )

# Collapse the options of each search into min / median / max price, per itinerary.
history = (
    options.groupby(["itinerary_id", "searched_at"])["price"]
    .agg(min_price="min", median_price="median", max_price="max", num_options="count")
    .reset_index()
    .sort_values(["itinerary_id", "searched_at"])
)
history

In [ ]:
def _label(itin_id):
    meta = itineraries.loc[itineraries["id"] == itin_id]
    if meta.empty:
        return f"itinerary {itin_id}"
    m = meta.iloc[0]
    label = (
        f"{itin_id}: {m['departure_id']}→{m['arrival_id']} "
        f"({m['travel_class']}, out {m['outbound_date']}"
    )
    if pd.notna(m["return_date"]):
        label += f", ret {m['return_date']}"
    return label + ")"


# y-axis currency label (itineraries are assumed to share a currency).
_currencies = itineraries["currency"].dropna()
currency = _currencies.iloc[0] if not _currencies.empty else ""
ylabel = f"price ({currency})" if currency else "price"

cmap = plt.get_cmap("tab10")
fig, axes = plt.subplots(
    len(ITINERARY_IDS),
    1,
    figsize=(11, 3.2 * len(ITINERARY_IDS)),
    sharex=True,
    squeeze=False,
)
axes = axes.ravel()

for i, (itin_id, ax) in enumerate(zip(ITINERARY_IDS, axes)):
    h = history[history["itinerary_id"] == itin_id]
    ax.set_title(_label(itin_id), fontsize=10)
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    if h.empty:
        ax.text(0.5, 0.5, "no data", ha="center", va="center", transform=ax.transAxes)
        continue
    # Cheapest fare per search.
    ax.plot(h["searched_at"], h["min_price"], marker="o", color=cmap(i % 10))

axes[-1].set_xlabel("search date")
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
fig.suptitle("Flight price history — cheapest fare per search")
fig.autofmt_xdate()
plt.tight_layout()
plt.show()